In [1]:
from neo4j import GraphDatabase

URI = "bolt://172.19.0.3/16:7687"
AUTH = ("neo4j", "12345678")

driver = GraphDatabase.driver(URI, auth=AUTH)

def execute_query(query):
    with driver.session() as session:
        result = session.run(query)
        return [record.data() for record in result]

In [2]:
query_criacao_streaming = """
MERGE (u1:Usuario {nome: 'Ana'})
MERGE (u2:Usuario {nome: 'Bruno'})
MERGE (u3:Usuario {nome: 'Carlos'})
MERGE (u4:Usuario {nome: 'Diana'})

MERGE (f1:Filme {titulo: 'Matrix'})
MERGE (g1:Genero {nome: 'Ficção Científica'})
MERGE (f1)-[:PERTENCE_AO]->(g1)

MERGE (f2:Filme {titulo: 'Interestelar'})
MERGE (f2)-[:PERTENCE_AO]->(g1)

MERGE (f3:Filme {titulo: 'Titanic'})
MERGE (g2:Genero {nome: 'Romance'})
MERGE (f3)-[:PERTENCE_AO]->(g2)

MERGE (f4:Filme {titulo: 'Gladiador'})
MERGE (g3:Genero {nome: 'Ação'})
MERGE (f4)-[:PERTENCE_AO]->(g3)

MERGE (f5:Filme {titulo: 'John Wick'})
MERGE (f5)-[:PERTENCE_AO]->(g3)

MERGE (u1)-[:ASSISTIU]->(f1)
MERGE (u1)-[:ASSISTIU]->(f2)
MERGE (u1)-[:ASSISTIU]->(f3)

MERGE (u2)-[:ASSISTIU]->(f1)

MERGE (u3)-[:ASSISTIU]->(f4)
MERGE (u3)-[:ASSISTIU]->(f2)

MERGE (u4)-[:ASSISTIU]->(f3)
MERGE (u4)-[:ASSISTIU]->(f1)
MERGE (u4)-[:ASSISTIU]->(f5)
"""
execute_query(query_criacao_streaming)

print("query listando todos filmes assistidos por ana")
query_ana = "MATCH (:Usuario {nome: 'Ana'})-[:ASSISTIU]->(f:Filme) RETURN f.titulo AS Filme"
for r in execute_query(query_ana):
    print(r)

print("query contando quantos usuarios assistiram cada filme")
query_total = """
MATCH (u:Usuario)-[:ASSISTIU]->(f:Filme)
RETURN f.titulo AS Filme, COUNT(u) AS TotalUsuarios
ORDER BY TotalUsuarios DESC
"""
for r in execute_query(query_total):
    print(r)

query listando todos filmes assistidos por ana
{'Filme': 'Titanic'}
{'Filme': 'Matrix'}
{'Filme': 'Interestelar'}
query contando quantos usuarios assistiram cada filme
{'Filme': 'Matrix', 'TotalUsuarios': 3}
{'Filme': 'Titanic', 'TotalUsuarios': 2}
{'Filme': 'Interestelar', 'TotalUsuarios': 2}
{'Filme': 'Gladiador', 'TotalUsuarios': 1}
{'Filme': 'John Wick', 'TotalUsuarios': 1}


In [3]:
query_criacao_planetas = """
MERGE (p1:Planeta {nome: 'Kepler-22b'})
MERGE (s1:Estrela {nome: 'Kepler-22'})
MERGE (o1:Observatorio {nome: 'Kepler Space Telescope'})
MERGE (p1)-[:ORBITA]->(s1)
MERGE (p1)-[:DESCOBERTO_POR]->(o1)

MERGE (p2:Planeta {nome: 'TRAPPIST-1b'})
MERGE (p3:Planeta {nome: 'TRAPPIST-1c'})
MERGE (s2:Estrela {nome: 'TRAPPIST-1'})
MERGE (o2:Observatorio {nome: 'La Silla Observatory'})
MERGE (p2)-[:ORBITA]->(s2)
MERGE (p2)-[:DESCOBERTO_POR]->(o2)
MERGE (p3)-[:ORBITA]->(s2)
MERGE (p3)-[:DESCOBERTO_POR]->(o2)

FOREACH (i IN [4,5,6,7,8,9,10] |
    MERGE (px:Planeta {nome: 'Planeta-X' + i})
    MERGE (sx:Estrela {nome: 'Estrela-Y' + (i % 3)})
    MERGE (ox:Observatorio {nome: 'Observatorio-Z' + (i % 2)})
    MERGE (px)-[:ORBITA]->(sx)
    MERGE (px)-[:DESCOBERTO_POR]->(ox)
)
"""
execute_query(query_criacao_planetas)

print("Planetas e Estrelas Hospedeiras")
query_lista = """
MATCH (p:Planeta)-[:ORBITA]->(e:Estrela)
RETURN p.nome AS Planeta, e.nome AS Estrela
LIMIT 10
"""
for r in execute_query(query_lista):
    print(r)

print("Contagem de planetas por estrela")
query_contagem = """
MATCH (p:Planeta)-[:ORBITA]->(e:Estrela)
RETURN e.nome AS Estrela, COUNT(p) AS TotalPlanetas
ORDER BY TotalPlanetas DESC
"""
for r in execute_query(query_contagem):
    print(r)

Planetas e Estrelas Hospedeiras
{'Planeta': 'Kepler-22b', 'Estrela': 'Kepler-22'}
{'Planeta': 'TRAPPIST-1b', 'Estrela': 'TRAPPIST-1'}
{'Planeta': 'TRAPPIST-1c', 'Estrela': 'TRAPPIST-1'}
{'Planeta': 'Planeta-X4', 'Estrela': 'Estrela-Y1'}
{'Planeta': 'Planeta-X5', 'Estrela': 'Estrela-Y2'}
{'Planeta': 'Planeta-X6', 'Estrela': 'Estrela-Y0'}
{'Planeta': 'Planeta-X7', 'Estrela': 'Estrela-Y1'}
{'Planeta': 'Planeta-X8', 'Estrela': 'Estrela-Y2'}
{'Planeta': 'Planeta-X9', 'Estrela': 'Estrela-Y0'}
{'Planeta': 'Planeta-X10', 'Estrela': 'Estrela-Y1'}
Contagem de planetas por estrela
{'Estrela': 'Estrela-Y1', 'TotalPlanetas': 3}
{'Estrela': 'TRAPPIST-1', 'TotalPlanetas': 2}
{'Estrela': 'Estrela-Y2', 'TotalPlanetas': 2}
{'Estrela': 'Estrela-Y0', 'TotalPlanetas': 2}
{'Estrela': 'Kepler-22', 'TotalPlanetas': 1}
